[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/yellow/notebooks/yellow_screening_analysis.ipynb)

# Screening the generated library against ACE1

**Yellow group · Hypertension**

The group generated 41,372 new molecules with REINVENT, all of them meant to be indoles or
xanthones. This notebook sorts them into those two families, looks at how the ACE1 model
scores each family, and shows the best-scoring molecules so a chemist can judge them.

## What you will do

- Load the generated library and set the two reference drugs aside.
- Label every molecule as an indole or a xanthone by searching for the ring system.
- Filter the library down to molecules with drug-like size, greasiness and QED.
- Compare how the ACE1 model scores the two families.
- Look at the 25 best-scoring molecules of each family and judge them as a chemist would.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "yellow"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the generated library

REINVENT is a **generative model**: instead of predicting something about a molecule, it
invents new ones. The group asked it for molecules built around two ring systems found in
natural products, and it returned 41,372 of them. None of these molecules has ever been
made or tested. They are ideas, and this notebook is about deciding which ideas are worth
anyone's time.

The file also holds two real ACE1 drugs, **captopril** and **lisinopril**. They were added
as a sanity check: a useful score should rate them highly.

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import chemspace, modelling

RANDOM_SEED = 42

generated = pd.read_csv("data/reinvent_mols_41k.csv")
print(f"{len(generated):,} molecules")
generated.head()

The two drugs are the only rows whose `compound_id` is not an `HJ_` code, so they are
easy to separate. Everything from here on treats them as references, not as candidates.

In [ ]:
is_reference = ~generated["compound_id"].str.startswith("HJ_")
references = generated[is_reference].reset_index(drop=True)
library = generated[~is_reference].reset_index(drop=True)

print(f"{len(library):,} generated molecules, {len(references)} reference drugs")
references

## 2. Label each molecule as an indole or a xanthone

The generator was asked for two kinds of molecule, but it does not attach a label saying
which is which. We have to work it out from the structure itself.

To do that we search each molecule for a **ring system**: the indole (a five-membered ring
containing a nitrogen, fused to a benzene ring) or the xanthone (two benzene rings joined
by an oxygen and by a carbonyl group). A ring system is written as a **SMARTS** pattern, a
short piece of text describing a partial molecule that can be searched for inside a whole
one.

In [ ]:
PATTERNS = {
    "indole": "c1ccc2c(c1)cc[nH0,nH]2",
    "xanthone": "O=c1c2ccccc2oc2ccccc12",
}

found = pd.DataFrame({name: chemspace.has_substructure(library["smiles"], smarts)
                      for name, smarts in PATTERNS.items()})
found.sum().to_frame("molecules containing it")

> **Note:** The indole pattern here is written to accept a nitrogen carrying a
> substituent (`[nH0,nH]`), not only a bare N-H. Writing it as a plain `[nH]`, which looks
> like the obvious choice and is what `yellow_chemical_space.ipynb` uses on the curated
> data, would miss about 6,900 of these molecules. A SMARTS pattern is always narrower
> than the idea a chemist has in mind, so it is worth checking what yours actually
> matches.

The two counts above add up to more than the number of molecules, because a few contain
**both** ring systems. A few others contain neither: the generator is not perfect. Neither
group belongs to one family, so we set both aside and keep only the molecules that match
exactly one pattern.

In [ ]:
one_family = found.sum(axis=1) == 1
dropped = (~one_family).sum()

library = library[one_family].copy()
library["family"] = np.where(found.loc[one_family, "indole"], "indole", "xanthone")

print(f"{dropped} molecules dropped: {(found.sum(axis=1) == 2).sum()} match both "
      f"patterns, {(found.sum(axis=1) == 0).sum()} match neither")
library["family"].value_counts().to_frame("molecules")

The same counts as a bar chart. The colours here (yellow for indoles, blue for
xanthones) are used for these two families everywhere in this notebook.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

FAMILIES = ["indole", "xanthone"]
COLORS = {"indole": nc.yellow, "xanthone": nc.blue}
counts = library["family"].value_counts().reindex(FAMILIES)

fig, axs = stylia.create_figure(1, 1, width=0.3, height=0.3)
ax = axs.next()
ax.bar(counts.index, counts.values, color=[COLORS[f] for f in FAMILIES])
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['indole'] / counts.sum():.0%} of the library is indoles")

Three examples of each family, so you can see what the labels mean. The indoles all
share the same fused two-ring core, and the xanthones the same three-ring core; everything
else around it is what the generator varied.

In [ ]:
examples = library.groupby("family").head(3)
chemspace.draw_molecules(examples["smiles"],
                         [f"{r.compound_id} ({r.family})" for r in examples.itertuples()],
                         per_row=3)

## 3. Filter for drug-like properties

A generative model invents molecules that satisfy its instructions, and nothing else. It
has no idea whether the result could be swallowed as a tablet, and plenty of what it
produced here is far too big or too greasy to work as a medicine.

Before spending model time and a chemist's attention on 40,871 molecules, we narrow the
library to the range that oral drugs occupy. We use three numbers:

- **Molecular weight (`mw`)**, roughly how big the molecule is, in g/mol. We keep
  **300 to 600**: below that a molecule rarely has room to bind tightly, above it
  absorption from the gut falls off.
- **logP**, how greasy the molecule is. High means it prefers fat, low means water. A drug
  has to do some of both, so we keep **below 5**.
- **QED**, or drug-likeness, a single score from 0 to 1 that combines those two with
  several other properties, each compared with the range seen in approved oral drugs. We
  keep **above 0.4**.

In [ ]:
MW_RANGE = (300, 600)
MAX_LOGP = 5
MIN_QED = 0.4

# This computes three properties for each of the 40,871 molecules and takes a minute.
library[["mw", "logp", "qed"]] = chemspace.molecular_properties(library["smiles"]).values
library[["mw", "logp", "qed"]].describe().round(2)

Before cutting anything, look at where the molecules actually sit. Each panel below is
one property, with the two families drawn on top of each other and the cutoffs as dashed
lines. Everything outside the dashed lines is about to be removed.

In [ ]:
PROPERTIES = {
    "mw": ("Molecular weight (g/mol)", (0, 900), MW_RANGE),
    "logp": ("logP", (-3, 10), (None, MAX_LOGP)),
    "qed": ("Drug-likeness (QED)", (0, 1), (MIN_QED, None)),
}

fig, axs = stylia.create_figure(1, 3)
for position, (column, (name, span, cutoffs)) in enumerate(PROPERTIES.items()):
    ax = axs.next()
    for family in FAMILIES:
        ax.hist(library.loc[library["family"] == family, column], bins=40, range=span,
                histtype="stepfilled", alpha=0.6, color=COLORS[family], label=family)
    for cutoff in cutoffs:
        if cutoff is not None:
            ax.axvline(cutoff, color=nc.pink, linestyle="--")
    if position == 0:
        ax.legend()
    stylia.label(ax, xlabel=name, ylabel="Molecules")

Now we apply the three rules together. A molecule has to pass all of them to stay.

In [ ]:
passes = (library["mw"].between(*MW_RANGE) & (library["logp"] < MAX_LOGP)
           & (library["qed"] > MIN_QED))

for name, rule in [("molecular weight", library["mw"].between(*MW_RANGE)),
                   ("logP", library["logp"] < MAX_LOGP),
                   ("QED", library["qed"] > MIN_QED)]:
    print(f"{name}: {rule.sum():,} pass ({rule.mean():.0%})")
print(f"all three: {passes.sum():,} of {len(library):,} ({passes.mean():.0%})")

library = library[passes].reset_index(drop=True)
library["family"].value_counts().to_frame("molecules kept")

The filter removes about two thirds of the library, and it removes the same share of
each family, so it does not favour indoles over xanthones.

It is worth checking a filter against molecules whose answer you already know. Here that
is captopril and lisinopril, the two ACE1 drugs.

In [ ]:
checked = references.copy()
checked[["mw", "logp", "qed"]] = chemspace.molecular_properties(references["smiles"]).values
checked["passes"] = (checked["mw"].between(*MW_RANGE) & (checked["logp"] < MAX_LOGP)
                     & (checked["qed"] > MIN_QED))
checked[["compound_id", "mw", "logp", "qed", "passes"]].round(2)

> **Note:** Both drugs fail. Captopril weighs 217 g/mol, well under the 300 lower
> bound, and lisinopril scores 0.38 on QED, just under 0.4. ACE1 inhibitors were designed
> to imitate a short piece of a protein, so they are smaller and more polar than the
> average oral drug, and a general drug-likeness filter suits them badly. This filter
> would have thrown away the very drugs the project is trying to improve on.
>
> That does not make the filter useless: it still removes the genuinely unusable molecules
> the generator produced. It does mean the thresholds are a choice, not a law, and this
> group has a good reason to question the lower weight bound in particular.

> **Exercise:** Lower `MW_RANGE` to `(200, 600)` and rerun from this section. How many
> more molecules survive, and does captopril pass now? Then decide which version you would
> defend to the rest of the group.

## 4. Load the ACE1 predictions

`yellow_baseline_models.ipynb` trained a LazyQSAR classifier on the curated ACE1 data and
saved it. Running that model over these molecules gives each one a **probability of being
an ACE1 inhibitor**, between 0 and 1.

Those predictions are made outside this notebook, uploaded to the group's Drive folder and
copied into `data/`, exactly like the curated data was. This notebook only reads them. The
file is expected to have a `compound_id` column and a `probability` column.

In [ ]:
SCORES_FILE = "data/lazyqsar_ace1_reinvent_41k.csv"
USING_PLACEHOLDER = not os.path.exists(SCORES_FILE)

print(f"{SCORES_FILE}: {'NOT FOUND' if USING_PLACEHOLDER else 'found'}")

> **Note: if the file was not found, the scores below are invented.** The real
> predictions are not ready yet. So that the rest of the notebook can be written and
> tested in the meantime, the cell below makes up a plausible-looking number for every
> molecule. These are random numbers. They say nothing whatsoever about ACE1, and every
> figure made from them is marked. Once the real file is in `data/`, this cell is skipped
> automatically, the marking disappears, and nothing else in the notebook changes.

In [ ]:
if USING_PLACEHOLDER:
    rng = np.random.default_rng(RANDOM_SEED)
    # A beta distribution stays between 0 and 1 on its own, so the invented scores do not
    # pile up at the edges the way a bell curve cut off at 0 would.
    shape = library["family"].map({"indole": 4.5, "xanthone": 3.0})
    scores = pd.DataFrame({
        "compound_id": list(library["compound_id"]) + list(references["compound_id"]),
        "probability": list(rng.beta(shape, 5.5)) + [0.92, 0.88]})
else:
    scores = pd.read_csv(SCORES_FILE)

TITLE_SUFFIX = " - PLACEHOLDER SCORES, NOT REAL" if USING_PLACEHOLDER else ""
SCORE_TAG = " (FAKE)" if USING_PLACEHOLDER else ""
scores.head()

Now we attach a score to each labelled molecule, matching the two tables on
`compound_id`. Any molecule without a score is reported and then left out, so that the
counts in the rest of the notebook stay honest.

In [ ]:
wanted = ["compound_id", "probability"] + (["rank"] if "rank" in scores.columns else [])
scored = library.merge(scores[wanted], on="compound_id", how="left")
reference_scores = references.merge(scores[wanted], on="compound_id", how="left")

print(f"{scored['probability'].notna().sum():,} of {len(scored):,} molecules have a score")
scored = scored.dropna(subset=["probability"]).reset_index(drop=True)
scored.head()

## 5. Score distributions by family

The first question is whether the model likes one family more than the other. A
**histogram** answers it: it counts how many molecules fall in each narrow band of score,
so the shape shows where each family sits.

The two dashed lines are captopril and lisinopril, the real drugs. They are the closest
thing we have to a target: a generated molecule scoring near them is, at least as far as
this model can tell, in the right range.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for family in FAMILIES:
    values = scored.loc[scored["family"] == family, "probability"]
    ax.hist(values, bins=40, range=(0, 1), histtype="stepfilled", alpha=0.6,
            color=COLORS[family], label=f"{family} (median {values.median():.2f})")
for row in reference_scores.itertuples():
    ax.axvline(row.probability, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="Predicted probability of inhibiting ACE1", ylabel="Molecules",
             title=f"Scores by family{TITLE_SUFFIX}")

The same thing as numbers. The last two columns are the share of each family the
model would call active at two different bars: 0.5, the usual cutoff, and 0.8, a stricter
one that gives a shortlist small enough to look at by hand.

In [ ]:
summary = scored.groupby("family")["probability"].agg(
    molecules="size", median="median",
    above_0_5=lambda s: (s >= 0.5).mean(), above_0_8=lambda s: (s >= 0.8).mean())
summary[["above_0_5", "above_0_8"]] = summary[["above_0_5", "above_0_8"]].map("{:.1%}".format)
summary.round(3)

Read these numbers carefully, and read them as a ranking rather than as a
prediction of what will happen in a laboratory.

The model was trained on about a thousand known ACE1 compounds, most of them peptide-like
molecules from a different part of chemical space than these indoles and xanthones. Asked
about a molecule unlike anything it has seen, it still returns a confident-looking number,
and that number has very little behind it. This is the **applicability domain** problem,
and `yellow_chemical_space.ipynb` is the notebook that measures it.

So a high score here means "worth a closer look", never "this works".

## 6. The top 25 of each family

Numbers can only take this so far. The last step is the oldest one in drug discovery:
look at the molecules.

We take the 25 best-scoring molecules of each family and draw them. Looking at them as a
group is what makes the problems visible, because the failures repeat.

In [ ]:
top_indoles = scored[scored["family"] == "indole"].nlargest(25, "probability")
chemspace.draw_molecules(
    top_indoles["smiles"],
    [f"{r.compound_id}: {r.probability:.2f}{SCORE_TAG}" for r in top_indoles.itertuples()],
    per_row=5)

And the 25 best xanthones.

In [ ]:
top_xanthones = scored[scored["family"] == "xanthone"].nlargest(25, "probability")
chemspace.draw_molecules(
    top_xanthones["smiles"],
    [f"{r.compound_id}: {r.probability:.2f}{SCORE_TAG}" for r in top_xanthones.itertuples()],
    per_row=5)

> **Exercise:** Go through the two grids and ask three questions of each one. Are the
> 25 molecules genuinely different from each other, or is the model rewarding one core
> over and over with tiny changes around it? Is there anything a chemist would refuse to
> make, such as a very strained ring or a group that would fall apart in water? And do the
> high scorers share a feature with the drugs in the training data, which would suggest
> the model has learned chemistry rather than an artefact?

Finally we save the whole scored and labelled table, so the group can open it in a
spreadsheet or take it to the next notebook. In Colab the file is deleted when the runtime
disconnects, so this downloads it to your computer as well.

In [ ]:
os.makedirs("outputs", exist_ok=True)
name = "yellow_screening_scored_placeholder" if USING_PLACEHOLDER else "yellow_screening_scored"
path = f"outputs/{name}.csv"

scored.to_csv(path, index=False)
modelling.download_file(path)

## Summary

- You loaded the 41,372 generated molecules, set the two reference drugs aside, and sorted
  the rest into indoles and xanthones by searching for the ring system. 40,871 molecules
  matched exactly one family; 501 were dropped, 444 because they contain both ring systems
  and 57 because they contain neither.
- The library is 60% indoles (24,603) and 40% xanthones (16,268).
- Filtering on molecular weight (300 to 600), logP (below 5) and QED (above 0.4) kept
  14,169 molecules, 8,359 indoles and 5,810 xanthones. That is about a third of each
  family, so the filter does not favour one over the other. Both reference drugs fail it,
  which says more about the filter than about them.
- You compared the score distributions of the two families, and looked at the 25
  best-scoring molecules of each.
- Any score in this notebook is a ranking to guide what to look at next, not evidence of
  activity: these molecules sit outside the chemical space the model was trained on.
- If section 3 reported the scores file as NOT FOUND, every score here is invented and the
  figures say so. Rerun the notebook once the real predictions are in `data/`.

**Next:** run `yellow_chemical_space.ipynb` on the top-scoring molecules to check whether
they fall inside the model's applicability domain, before anyone spends time on them.